<a href="https://colab.research.google.com/github/hamsamahmoud-cpu/DistilBERT-Fine-tuning-for-Movie-Review-Sentiment-Analysis1/blob/main/04_evaluation_and_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers datasets evaluate accelerate

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer
from datasets import load_from_disk
from google.colab import drive  # <-- Import the drive library

# 0. Mount Google Drive to connect this session to your files
drive.mount('/content/drive')

# 1. Setup Paths
PROJECT_PATH = '/content/drive/MyDrive/distilbert-imdb-sentiment'
DATA_PATH = os.path.join(PROJECT_PATH, "tokenized_data")
MODEL_PATH = os.path.join(PROJECT_PATH, "distilbert-finetuned")

# 2. Load Data and Model
print("Loading data and model from Drive...")
tokenized_datasets = load_from_disk(DATA_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("Successfully loaded!")

# 3. Create a test subset for evaluation
test_data = tokenized_datasets["test"].select(range(1000))

# 4. Get Predictions using the Trainer API
trainer = Trainer(model=model)
predictions_output = trainer.predict(test_data)
predictions = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer
from datasets import load_from_disk

# 1. Load Data and Model
PROJECT_PATH = '/content/drive/MyDrive/distilbert-imdb-sentiment'
DATA_PATH = os.path.join(PROJECT_PATH, "tokenized_data")
MODEL_PATH = os.path.join(PROJECT_PATH, "distilbert-finetuned")

tokenized_datasets = load_from_disk(DATA_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Shuffle the test dataset BEFORE selecting the 1000 sample slice.
# The seed ensures that your evaluation subset remains reproducible across runs.
test_data = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

# 2. Get Predictions using the Trainer API
trainer = Trainer(model=model)
predictions_output = trainer.predict(test_data)

# Extract predicted token indices and true ground truth labels
predictions = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

# 3. Generate and Plot the Balanced Confusion Matrix
print("\n Generating balanced Confusion Matrix...")
cm = confusion_matrix(true_labels, predictions)

# Display matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')
plt.title('DistilBERT Evaluation Confusion Matrix (Balanced Sample)', fontweight='bold')

# Save directly to your working directory or Google Drive for your paper layout
plt.savefig(os.path.join(PROJECT_PATH, 'balanced_confusion_matrix.png'), bbox_inches='tight', dpi=300)
plt.show()

# 4. Print Summary Statistics to verify metrics
print("\n Evaluation Metrics Report:")
print(classification_report(true_labels, predictions, target_names=['Negative', 'Positive']))

In [ ]:
# Reconstruct the original text to see what confused the model
test_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in test_data["input_ids"]]

results_df = pd.DataFrame({
    "Review": test_texts,
    "True_Label": true_labels,
    "Prediction": predictions
})

# False Positives: Model guessed Positive (1), but it was actually Negative (0)
false_positives = results_df[(results_df["Prediction"] == 1) & (results_df["True_Label"] == 0)]

# False Negatives: Model guessed Negative (0), but it was actually Positive (1)
false_negatives = results_df[(results_df["Prediction"] == 0) & (results_df["True_Label"] == 1)]

print("--- TOP 3 FALSE POSITIVES (Sarcasm? Mixed reviews?) ---")
for text in false_positives["Review"].head(3):
    print(f"- {text[:200]}...\n")

print("--- TOP 3 FALSE NEGATIVES (Too subtle?) ---")
for text in false_negatives["Review"].head(3):
    print(f"- {text[:200]}...\n")

In [ ]:
from transformers import TrainingArguments
import evaluate
import time

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

# We will use a baseline pool of 4,000 reviews

full_train_pool = tokenized_datasets["train"].shuffle(seed=42).select(range(4000))
eval_subset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

fractions = [0.10, 0.25, 0.50, 1.0]
ablation_results = {"Size": [], "Accuracy": [], "Time_Seconds": []}

for frac in fractions:
    num_samples = int(len(full_train_pool) * frac)
    train_subset = full_train_pool.select(range(num_samples))

    print(f"\n--- Training on {frac*100}% of data ({num_samples} samples) ---")

    # Reload a fresh, untrained model for a fair test
    fresh_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    for layer in fresh_model.distilbert.transformer.layer[:4]:
        for param in layer.parameters():
            param.requires_grad = False

    training_args = TrainingArguments(
        output_dir=f"./results_{frac}",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        report_to="none"
    )

    trainer = Trainer(
        model=fresh_model,
        args=training_args,
        train_dataset=train_subset,
        eval_dataset=eval_subset,
        compute_metrics=compute_metrics,
    )

    start_time = time.time()
    trainer.train()
    end_time = time.time()

    # Evaluate and store results
    metrics = trainer.evaluate()

    ablation_results["Size"].append(f"{frac*100}%")
    ablation_results["Accuracy"].append(metrics["eval_accuracy"])
    ablation_results["Time_Seconds"].append(end_time - start_time)

print("\n=== Ablation Study Complete ===")
results_df = pd.DataFrame(ablation_results)
print(results_df)

In [ ]:
# Plot Accuracy vs. Data Size
fig, ax1 = plt.subplots(figsize=(8, 5), dpi=300)

color = 'tab:blue'
ax1.set_xlabel('Training Data Size')
ax1.set_ylabel('Accuracy', color=color)
ax1.plot(results_df["Size"], results_df["Accuracy"], marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

# Create a second y-axis for Training Time
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Training Time (Seconds)', color=color)
ax2.plot(results_df["Size"], results_df["Time_Seconds"], marker='s', color=color, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title("Ablation Study: Model Accuracy and Training Time vs. Data Size")
fig.tight_layout()
plt.show()